# ExtendingMemoryLLM — Colab Setup & Experiment Runner

**Before running:** Runtime → Change runtime type → **A100 GPU**

This notebook:
1. Checks GPU and mounts Drive for persistent storage
2. Clones the repo and installs dependencies
3. Logs into HuggingFace and downloads eval data
4. Runs a smoke test
5. Runs full retention eval experiments

**Run cells 1–4 once per session. Cell 5+ can be re-run to launch experiments.**

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())

import torch
print(f'PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Mount Google Drive (for persistent results storage) ───────────
from google.colab import drive
drive.mount('/content/drive')

# Results will be saved here — survives session restarts
RESULTS_DIR = '/content/drive/MyDrive/ExtendingMemoryLLM/results'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

In [ ]:
# ── Cell 3: Clone repo & install dependencies ─────────────────────────────
import os

REPO_DIR = '/content/ExtendingMemoryLLM'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/akshatbhandari15/ExtendingMemoryLLM.git $REPO_DIR
else:
    print('Repo already cloned — pulling latest...')
    !cd $REPO_DIR && git pull

%cd $REPO_DIR

# Install dependencies
!bash scripts/setup.sh --no-data  # --no-data skips data download (handled in Cell 4)

In [ ]:
# ── Cell 4: Download eval data ────────────────────────────────────────────
# SQuAD v2
import os
os.makedirs('data/squad', exist_ok=True)
os.makedirs('data/nq', exist_ok=True)

if not os.path.exists('data/squad/dev-v2.0.json'):
    !wget -q https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json -O data/squad/dev-v2.0.json
    print('Downloaded SQuAD v2 dev set')
else:
    print('SQuAD dev set already present')

# Index files from HuggingFace
from huggingface_hub import hf_hub_download
import shutil

for fname, dest in [
    ('indices_squad_3.npy', 'data/squad/indices_squad_3.npy'),
    ('indices_nq_4.npy',    'data/nq/indices_nq_4.npy'),
]:
    if not os.path.exists(dest):
        src = hf_hub_download(repo_id='YuWangX/KnowledgeRetention',
                              filename=fname, repo_type='dataset')
        shutil.copy(src, dest)
        print(f'Downloaded {fname}')
    else:
        print(f'{fname} already present')

# NaturalQA: must be downloaded manually
if not os.path.exists('data/nq/v1.0-simplified_nq-dev-all.jsonl'):
    print('\nWARNING: NaturalQA file missing.')
    print('Download from: https://ai.google.com/research/NaturalQuestions/download')
    print('Upload to: data/nq/v1.0-simplified_nq-dev-all.jsonl')
    print('(or copy from Drive if already downloaded)')
else:
    print('NaturalQA data present')

print('\nData check complete.')

In [ ]:
# ── Cell 5: Smoke test — verify everything works (runs in ~2 min) ─────────
# 5 examples, 3 distractor steps, random strategy
# Expected: no errors, results saved, accuracy printed
!python run_eval.py \
    --strategy random \
    --dataset squad \
    --nuc 3 \
    --num_samples 5 \
    --output_dir $RESULTS_DIR

In [ ]:
# ── Cell 6: Run full retention eval — SQuAD v2, all strategies ───────────
# ~30-60 min total on A100 depending on strategy
# --resume means it skips strategies already completed (safe to re-run)
!python run_eval.py \
    --strategy all \
    --dataset squad \
    --nuc 20 \
    --num_samples 100 \
    --output_dir $RESULTS_DIR \
    --resume

In [ ]:
# ── Cell 7: Run full retention eval — NaturalQA, all strategies ──────────
!python run_eval.py \
    --strategy all \
    --dataset nq \
    --nuc 20 \
    --num_samples 100 \
    --output_dir $RESULTS_DIR \
    --resume

In [ ]:
# ── Cell 8: Quick results summary ─────────────────────────────────────────
import json, os, glob

files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.json')))
print(f'{'File':<40} {'AUC':>6}  {'Step-0':>7}  {'Step-20':>8}')
print('-' * 65)
for f in files:
    d = json.load(open(f))
    accs = d['accuracy_per_step']
    name = os.path.basename(f).replace('.json', '')
    step0  = f"{accs[0]:.3f}" if len(accs) > 0  else 'N/A'
    stepN  = f"{accs[-1]:.3f}" if len(accs) > 1 else 'N/A'
    print(f'{name:<40} {d["auc"]:>6.3f}  {step0:>7}  {stepN:>8}')